# Reproduce main-paper Tables 5 and 6

This release notebook reconstructs Tables 5 and 6 for every model reported in the paper. It reads the evaluated metric JSON files from `output/metrics`, combines the two VISA subsets, preserves the camera-ready model order, and prints both pandas and LaTeX tables.

Run the inference, answer-processing, and evaluation stages before executing this notebook.

## Expected metric outputs

Run `jobs/processing/metrics_processing.sh` before this notebook. The evaluation stage writes:

```text
output/metrics/
├── answers/    # answer metrics for each dataset, model, and task
└── iou/        # bounding-box metrics for each dataset, model, and task
```

Both directories contain JSON files whose names preserve the corresponding inference task: `answerBox`, `box`, or `boxFromAnswer`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the DocAttriBench repository.')


REPO_ROOT = find_repo_root()
METRICS_DIR = REPO_ROOT / 'output' / 'metrics'
ANSWER_DIR = METRICS_DIR / 'answers'
IOU_DIR = METRICS_DIR / 'iou'
THRESHOLD = 0.5
DATASETS = ['docvqa', 'visualmrc', 'visa', 'slidevqa', 'visualwebbench', 'longdocurl', 'mmlongbenchdoc']
VISA_PARTS = ('papervisa', 'wikivisa')

# Public output names plus legacy checkpoint names used for the camera-ready results.
PAPER_MODELS = {
    'InternVL2.5-2B': ('OpenGVLab_InternVL2_5-2B',),
    'InternVL3-2B': ('OpenGVLab_InternVL3-2B-Instruct',),
    'InternVL3.5-2B': ('OpenGVLab_InternVL3_5-2B-Instruct',),
    'Qwen2.5-VL-3B': ('Qwen_Qwen2.5-VL-3B-Instruct',),
    'Qwen3-VL-2B': ('Qwen_Qwen3-VL-2B-Instruct',),
    'InternVL2.5-8B': ('OpenGVLab_InternVL2_5-8B',),
    'InternVL3-8B': ('OpenGVLab_InternVL3-8B-Instruct',),
    'InternVL3.5-8B': ('OpenGVLab_InternVL3_5-8B-Instruct',),
    'Qwen2.5-VL-7B': ('Qwen_Qwen2.5-VL-7B-Instruct',),
    'Qwen3-VL-8B': ('Qwen_Qwen3-VL-8B-Instruct',),
    'InternVL2.5-38B': ('OpenGVLab_InternVL2_5-38B',),
    'InternVL3-38B': ('OpenGVLab_InternVL3-38B-Instruct',),
    'InternVL3.5-38B': ('OpenGVLab_InternVL3_5-38B-Instruct',),
    'Qwen2.5-VL-32B': ('Qwen_Qwen2.5-VL-32B-Instruct',),
    'Qwen3-VL-32B': ('Qwen_Qwen3-VL-32B-Instruct',),
    'VISA-7B': ('MrLight_visa-7B-single-fulldata-merged',),
    'MAPPET-7B': (
        'Mappet_Mappet-7B',
        'Mappet_Mappet2.5-VL-7B-Instruct-LoRA_v3_chkp_8400',
    ),
    'MAPPET-8B': (
        'Mappet_Mappet-8B',
        'Mappet_Mappet3-VL-8B-Instruct-LoRA_v3_chkp_11600',
    ),
    'MAPPET-I-8B': (
        'Mappet_Mappet-I-8B',
        'Mappet_MappetIntern3.5-VL-8B-Instruct-LoRA_v2_chkp_7600',
    ),
}
MODEL_ORDER = list(PAPER_MODELS)
MODEL_LABELS = {
    artifact: label
    for label, artifacts in PAPER_MODELS.items()
    for artifact in artifacts
}

In [ ]:
required_directories = {
    'answer metrics': ANSWER_DIR,
    'bounding-box metrics': IOU_DIR,
}
missing_directories = [path for path in required_directories.values() if not path.is_dir()]
empty_directories = [
    path for path in required_directories.values()
    if path.is_dir() and not any(path.glob('*.json'))
]
if missing_directories or empty_directories:
    problems = [*(f'missing: {path}' for path in missing_directories),
                *(f'no JSON files: {path}' for path in empty_directories)]
    details = '\n'.join(f'  - {problem}' for problem in problems)
    raise FileNotFoundError(
        'Metric outputs are not ready. Run jobs/processing/metrics_processing.sh first.\n'
        f'{details}'
    )

print(f'Loading metric outputs from: {METRICS_DIR}')

In [ ]:
def threshold_metrics(path):
    with path.open() as handle:
        rows = json.load(handle)
    return next(row for row in rows if row['thr'] == THRESHOLD)


def parse_path(path, suffix):
    dataset, rest = path.stem.split('_', 1)
    return dataset, rest[:-len(suffix)]


def merge_visa(frame, values):
    visa = frame[frame.dataset.isin(VISA_PARTS)].groupby('model', as_index=False)[values].mean()
    visa['dataset'] = 'visa'
    return pd.concat([frame[~frame.dataset.isin(VISA_PARTS)], visa], ignore_index=True)


def prefer_clean(rows):
    selected = {}
    for path, row in rows:
        key = (row['dataset'], row['model'])
        if key not in selected or path.name.endswith('_clean.json'):
            selected[key] = row
    return list(selected.values())


def select_paper_models(frame):
    frame = frame[frame.model.isin(MODEL_LABELS)].copy()
    frame['Model'] = frame.model.map(MODEL_LABELS)
    return frame


def task_f1(suffixes):
    rows = []
    for path in sorted(IOU_DIR.glob('*.json')):
        if not path.name.endswith(suffixes):
            continue
        suffix = next(value[:-5] for value in suffixes if path.name.endswith(value))
        dataset, model = parse_path(path, suffix)
        rows.append((path, {
            'dataset': dataset,
            'model': model,
            'f1': threshold_metrics(path)['macro_f1'] * 100,
        }))
    return select_paper_models(merge_visa(pd.DataFrame(prefer_clean(rows)), ['f1']))


def wide(frame, values, average=None):
    result = frame.pivot(index='Model', columns='dataset', values=values)
    result = result.reindex(index=MODEL_ORDER, columns=pd.MultiIndex.from_product([values, DATASETS]))
    if average:
        for value in average:
            result[('Avg', value)] = result[value].mean(axis=1)
    return result.round(1)

## Table 5: grounded answer generation

In [ ]:
rows = []
for answer_path in sorted(ANSWER_DIR.glob('*.json')):
    if not answer_path.name.endswith(('_answerBox.json', '_answerBox_clean.json')):
        continue
    suffix = '_answerBox_clean' if answer_path.name.endswith('_answerBox_clean.json') else '_answerBox'
    dataset, model = parse_path(answer_path, suffix)
    iou_path = IOU_DIR / answer_path.name
    if not iou_path.exists():
        continue
    answer, box = threshold_metrics(answer_path), threshold_metrics(iou_path)
    pairs = list(zip(answer.get('avg_scores', answer.get('per_answer_score', [])), box.get('all_f1s', [])))
    hybrid = sum(acc > THRESHOLD and f1 > THRESHOLD for acc, f1 in pairs) / len(pairs) if pairs else float('nan')
    rows.append({
        'dataset': dataset,
        'model': model,
        'Acc_txt': answer['macro_acc'] * 100,
        'F1_box': box['macro_f1'] * 100,
        'Acc': hybrid * 100,
    })

table5_source = select_paper_models(merge_visa(pd.DataFrame(rows), ['Acc_txt', 'F1_box', 'Acc']))
table5 = wide(table5_source, ['Acc_txt', 'F1_box', 'Acc'], average=['Acc'])
display(table5)
print(table5.to_latex(escape=False, na_rep='-'))

## Table 6: answer locating and post-hoc attribution

In [ ]:
locating = task_f1(('_box.json', '_box_clean.json')).rename(columns={'f1': 'F1_box_Q'})
posthoc = task_f1(('_boxFromAnswer.json', '_boxFromAnswer_clean.json')).rename(columns={'f1': 'F1_box_QA'})
table6_source = locating.merge(posthoc[['dataset', 'model', 'F1_box_QA']], on=['dataset', 'model'], how='outer')
table6_source['Model'] = table6_source.model.map(MODEL_LABELS)
table6 = wide(table6_source, ['F1_box_Q', 'F1_box_QA'], average=['F1_box_Q', 'F1_box_QA'])
display(table6)
print(table6.to_latex(escape=False, na_rep='-'))
